# Translation Widget
Powered by Meta's NLLB-200 — 200 languages, dialect-aware.
To change the language pair, only edit the **Config** cell.

In [1]:
# ── Config ──────────────────────────────────────────────────────────────────
# NLLB language codes: https://github.com/facebookresearch/flores/blob/main/flores200/README.md
#
# Common codes:
#   Arabic        → arb_Arab
#   English       → eng_Latn
#   French        → fra_Latn
#   Spanish       → spa_Latn
#   German        → deu_Latn

SRC_LANG   = "arb_Arab"   # source language code
TGT_LANG   = "eng_Latn"   # target language code
SRC_LABEL  = "Arabic"     # display label
TGT_LABEL  = "English"    # display label

In [2]:
# ── Setup ────────────────────────────────────────────────────────────────────
# !uv pip install transformers sentencepiece torch ipywidgets

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import ipywidgets as widgets
from IPython.display import display, HTML

print("Loading model (downloads ~1.2 GB on first run) …")
tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
model     = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M")
print("Ready.")

Loading model (downloads ~1.2 GB on first run) …


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Ready.


In [3]:
# ── Widget ───────────────────────────────────────────────────────────────────
display(HTML("""
<style>
  .tr-card {
    background:#1e1e2e; border-radius:12px; padding:20px 24px;
    max-width:680px; font-family:'Inter',sans-serif;
    box-shadow:0 4px 20px rgba(0,0,0,.4);
  }
  .tr-card h3 { margin:0 0 4px; color:#cdd6f4; font-size:1.1rem; }
  .tr-card p  { margin:0 0 16px; color:#6c7086; font-size:.8rem; }
  .badge {
    display:inline-block; padding:2px 10px; border-radius:20px;
    font-size:.72rem; font-weight:700; letter-spacing:.05em;
    text-transform:uppercase; margin-bottom:5px;
  }
  .badge-src { background:#313244; color:#89dceb; }
  .badge-tgt { background:#313244; color:#a6e3a1; }
  .out-box {
    background:#181825; border:1px solid #45475a; border-radius:8px;
    padding:12px 14px; color:#cdd6f4; min-height:52px;
    font-size:1rem; white-space:pre-wrap;
  }
  .placeholder { color:#45475a; font-style:italic; }
</style>
"""))

header    = widgets.HTML(f'<div class="tr-card"><h3>🌐 {SRC_LABEL} → {TGT_LABEL}</h3><p>facebook/nllb-200-distilled-600M</p></div>')
src_badge = widgets.HTML(f'<span class="badge badge-src">{SRC_LABEL}</span>')
tgt_badge = widgets.HTML(f'<span class="badge badge-tgt">{TGT_LABEL}</span>')

input_box = widgets.Textarea(
    placeholder=f"Type {SRC_LABEL} here…",
    layout=widgets.Layout(width="100%", height="96px"),
)
btn_translate = widgets.Button(
    description="Translate",
    layout=widgets.Layout(width="110px", height="34px"),
    style=widgets.ButtonStyle(button_color="#89b4fa", font_weight="bold"),
)
btn_clear = widgets.Button(
    description="Clear",
    layout=widgets.Layout(width="72px", height="34px"),
    style=widgets.ButtonStyle(button_color="#45475a"),
)
status = widgets.Label(value="")
output = widgets.HTML('<div class="out-box placeholder">Translation appears here.</div>')

def on_translate(_):
    text = input_box.value.strip()
    if not text:
        status.value = "⚠️ Enter some text first."
        return
    status.value = "Translating…"
    btn_translate.disabled = True
    try:
        inputs    = tokenizer(text, return_tensors="pt")
        tgt_id    = tokenizer.convert_tokens_to_ids(TGT_LANG)
        outputs   = model.generate(**inputs, forced_bos_token_id=tgt_id, max_new_tokens=128)
        result    = tokenizer.decode(outputs[0], skip_special_tokens=True)
        output.value = f'<div class="out-box">{result}</div>'
        status.value = "✓ Done"
    except Exception as e:
        output.value = f'<div class="out-box" style="color:#f38ba8">Error: {e}</div>'
        status.value = ""
    finally:
        btn_translate.disabled = False

def on_clear(_):
    input_box.value = ""
    output.value = '<div class="out-box placeholder">Translation appears here.</div>'
    status.value = ""

btn_translate.on_click(on_translate)
btn_clear.on_click(on_clear)

display(widgets.VBox([
    header,
    src_badge, input_box,
    widgets.HBox([btn_translate, btn_clear, status],
                 layout=widgets.Layout(align_items="center", gap="8px", margin="6px 0")),
    tgt_badge, output,
], layout=widgets.Layout(max_width="680px", gap="3px")))